In [ ]:
! python3 -m pip install matplotlib
! python3 -m pip install qiskit qiskit-aer
import numpy as np
import matplotlib.pyplot as plt
from math import pi, gcd
from qiskit import *
from qiskit.circuit import *
from qiskit.circuit.library import *
from qiskit.quantum_info.operators import Operator
from qiskit_aer import AerSimulator, StatevectorSimulator
from scipy import optimize
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

Small library for pretty-printing (nothing to do)

In [ ]:
def nat2bl(pad,n):
    if n == 0: r = [0 for i in range(pad)]
    elif n % 2 == 1: r = nat2bl(pad-1,(n-1)//2); r.append(1)
    else: r = nat2bl(pad-1,n//2); r.append(0)
    return r

def bl2nat(s):
    if len(s) == 0: return 0
    else: a = s.pop(); return (a + 2*bl2nat(s))

def bl2bs(l):
    if len(l) == 0: return ""
    else: a = l.pop(); return (bl2bs(l) + str(a))

def nat2bs(pad,i): return bl2bs(nat2bl(pad,i))

def bs2bl(s):
    l = []
    for i in range(len(s)): l.append(int(s[i]))
    return l

def bs2nat(s): return bl2nat(bs2bl(s))

def processOneState(st): # Length = power of 2
        s = list(st)
        if len(s) == 2: return {'0' : s[0], '1' : s[1]}
        else:
            a0 = processOneState(s[:len(s)//2])
            a1 = processOneState(s[len(s)//2:])
            r = {}
            for k in a0: r['0' + k] = a0[k]
            for k in a1: r['1' + k] = a1[k]
            return r

def printOneState(d): # get a dict as per processStates output
    for k in d:
        im = d[k].imag
        re = d[k].real
        if abs(im) >= 0.001 or abs(re) >= 0.001:
            print("% .3f + % .3fj |%s>" % (re,im,k))

def printFinalRes(result):
    printOneState(processOneState(list(np.asarray(result))))

def runStateVector(qc):
    simulator = StatevectorSimulator()
    job = simulator.run(qc.decompose(reps=6), memory=True)
    job_result = job.result()
    result = job_result.results[0].to_dict()['data']['statevector']
    printFinalRes(result)

def runStateVectorSeveralTimes(qc, howmany):
    qc.save_statevector(label = 'collect', pershot = True)
    simulator = StatevectorSimulator()
    job = simulator.run(qc.decompose(reps=6), memory=True, shots=howmany)
    result = job.result()
    memory = result.data(0)['memory']
    collect = result.data(0)['collect']
    r = {}
    for i in range(len(collect)):
        r[str(collect[i])] = (0, collect[i])
    for i in range(len(collect)):
        n, v = r[str(collect[i])]
        r[str(collect[i])] = (n+1, v)
    for k in r:
        i, v = r[k]
        print(f"With {i} occurences:")
        printFinalRes(v)

def plotDistrib(d):
    sorted_items = sorted(d.items())
    keys = [k for k, _ in sorted_items]
    values = [v for _, v in sorted_items]
    plt.figure()
    plt.bar(keys, values)
    plt.xticks(rotation=90)
    plt.show()

def getSample(qc,howmany):
    simulator = AerSimulator()
    job = simulator.run(qc.decompose(reps=6), shots=howmany)
    res = dict(job.result().get_counts(qc))
    return res

def plotSample(qc,howmany):
    d = getSample(qc,howmany)
    ld = len(list(d.keys())[0])
    for i in range(2 ** ld):
        s = nat2bs(ld,i)
        if s not in d: d[s] = 0
    plotDistrib(d)

def entangledState(n):
    q = QuantumRegister(n, name="q")   # We need n qubits...
    c = ClassicalRegister(n, name="c") # ... and n bits to store the results of the measurement later on
    qc = QuantumCircuit(q,c) # the circuit !
    qc.h(0)
    for i in range(1,n):
        qc.cx(0,i)
    return qc

def QPE(U, size_eig=3, endMeasure=False):
    size_phi = 2
    eig = QuantumRegister(size_eig, name="eig")
    phi = QuantumRegister(size_phi, name="phi")
    ceig = ClassicalRegister(size_eig, name="ceig")
    qc = QuantumCircuit(eig, phi ,ceig)
    qc.h(eig)
    qc.x(phi)
    qc.append(U.control(), [eig[0]] + list(phi))
    for i in range(1,size_eig):
        qc.append(U.power(2 ** i).control(),[eig[i]] + list(phi))
    qc.barrier()
    qc.append(QFTGate(size_eig).inverse(),eig)
    qc.barrier()
    if endMeasure:
        qc.measure(eig,ceig)
    return qc

def QPEotherPhi(U, size_eig=3, endMeasure=False):
    size_phi = 2
    eig = QuantumRegister(size_eig, name="eig")
    phi = QuantumRegister(size_phi, name="phi")
    ceig = ClassicalRegister(size_eig, name="ceig")
    qc = QuantumCircuit(eig, phi ,ceig)
    qc.h(eig)
    qc.h(phi[0])
    qc.cx(phi[0],phi[1])
    qc.append(U.control(), [eig[0]] + list(phi))
    for i in range(1,size_eig):
        qc.append(U.power(2 ** i).control(),[eig[i]] + list(phi))
    qc.barrier()
    qc.append(QFTGate(size_eig).inverse(),eig)
    qc.barrier()
    if endMeasure:
        qc.measure(eig,ceig)
    return qc

def U(frac):
    return UnitaryGate(
        Operator([[1,0,0,0],
                  [0,1,0,0],
                  [0,0,1,0],
                  [0,0,0,np.exp(pi*2j*frac)]]), label="U")
def V(frac1, frac2):
    return UnitaryGate(
        Operator([[np.exp(pi*2j*frac1),0,0,0],
                  [0,1,0,0],
                  [0,0,1,0],
                  [0,0,0,np.exp(pi*2j*frac2)]]), label="U")
U = UnitaryGate(
    Operator([[1,0,0,0],
              [0,1,0,0],
              [0,0,1,0],
              [0,0,0,np.exp(pi*2j*(6/8))]]), label="U")
print("2 qubits")
runStateVector(entangledState(2))
print("4 qubits")
runStateVector(entangledState(4))
print("8 qubits")
runStateVector(entangledState(8))
print("16 qubits")
runStateVector(entangledState(16))
qc = entangledState(5)
q = qc.qregs[0]
c = qc.cregs[0]
qc.measure(q,c)
print(getSample(qc,1000))
print("The resulting state vector is")
runStateVector(QPE(U, size_eig=3))
print()
print("Measuring 'eig' yields the probabilistic distribution")
print(getSample(QPE(U, size_eig=3,endMeasure=True), 1000))
plotSample(QPE(U, size_eig=3,endMeasure=True), 1000)
for i in range(8):
    print(f"The fraction in the phase is now {i}/8, with 5 bits we get the result")
    print(getSample(QPE(U(i/8), size_eig=5, endMeasure=True), 1000))
for i in range(2,6):
    print(f"with {i} bits of precision")
    plotSample(QPE(U(1/3), size_eig=i,endMeasure=True), 1000)
for i in range(2,6):
    print(f"with {i} bits of precision")
    plotSample(QPEotherPhi(U(1/3), size_eig=i,endMeasure=True), 1000)
for i in range(2,6):
    print(f"with {i} bits of precision")
    plotSample(QPEotherPhi(V(1/3,2/3), size_eig=i,endMeasure=True), 1000)